# 02 — Pre-procés + segmentació de caràcters

Prototip del pas previ a l'OCR. Partim dels **retalls candidats** generats pel detector (`data/processed/*.png`), que poden contenir una matrícula real o ser un fals positiu. Per cada retall volem:

1. **Pre-processar** (grayscale, denoise, contrast, deskew).
2. **Binaritzar** (caràcters blancs sobre fons negre).
3. **Segmentar** en caràcters individuals (projecció vertical + fallback per components connexos).
4. **Validar i normalitzar** cada caràcter a mida fixa (64×32) per al futur classificador.
5. **Filtrar fals positius**: si la segmentació no troba entre 5 i 8 caràcters consistents, descartem el retall.

Aquest notebook **no fa classificació encara** — produeix els caràcters normalitzats que alimentaran el SVM/CNN al següent notebook.

In [ ]:
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

random.seed(0)
np.random.seed(0)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'

CROPS_DIR = Path('data/processed')
crops = sorted(CROPS_DIR.glob('*.png'))
print(f'OpenCV {cv2.__version__} — {len(crops)} crops a {CROPS_DIR}')

## 1. Inspecció ràpida dels retalls

Mirem una mostra aleatòria per fer-nos una idea de la varietat: alguns són matrícules netes, altres són trossos de carrosseria, fanals o ombres (fals positius del detector).

In [ ]:
N_SHOW = 12
sample_paths = random.sample(crops, N_SHOW)

fig, axes = plt.subplots(3, 4, figsize=(16, 7))
for ax, p in zip(axes.flat, sample_paths):
    img = cv2.imread(str(p))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Pre-procés

Tres operacions:

- **Up-scaling** si el retall és més baix que 64 px → la segmentació per projecció és més estable amb una alçada decent.
- **Bilateral filter**: suavitza soroll sense difuminar les vores dels caràcters.
- **CLAHE**: contrast local — imprescindible perquè les condicions d'il·luminació varien entre fotos.

In [ ]:
def preprocess(crop_bgr):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY) if crop_bgr.ndim == 3 else crop_bgr.copy()

    # Up-scaling si la ROI és massa petita
    h, w = gray.shape
    if h < 64:
        scale = 64 / h
        gray = cv2.resize(gray, (int(w * scale), 64), interpolation=cv2.INTER_CUBIC)

    # Denoise preservant vores
    gray = cv2.bilateralFilter(gray, d=7, sigmaColor=50, sigmaSpace=50)

    # Contrast local
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    return gray

## 3. Deskew (correcció d'inclinació)

Els retalls solen tenir una inclinació petita perquè la matrícula real no està perfectament horitzontal a la foto original. Trobem el contorn més gran (assumint que és la placa), li calculem el rectangle mínim amb `minAreaRect` i rotem.

Si el contorn és massa petit (< 5 % del retall) o l'angle és exagerat (> 15°), no rotem — probablement el retall no és una matrícula i rotar-lo empitjoraria les coses.

In [ ]:
def deskew(gray):
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return gray

    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 0.05 * gray.size:
        return gray

    (cx, cy), (w, h), angle = cv2.minAreaRect(largest)
    # Convenció OpenCV: angle ∈ (-90, 0]
    if w < h:
        angle += 90.0
    if abs(angle) > 15:
        return gray

    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    h_img, w_img = gray.shape
    return cv2.warpAffine(gray, M, (w_img, h_img), borderMode=cv2.BORDER_REPLICATE)

## 4. Binarització adaptativa

`adaptiveThreshold` Gaussià: el llindar es calcula per blocs locals, robust a ombres i degradats d'il·luminació. Resultat: caràcters **blancs** sobre fons **negre** (`THRESH_BINARY_INV`).

Després apliquem un *opening* 2×2 per netejar soroll puntiforme (motes de pols, JPEG artifacts) sense menjar-nos els caràcters.

In [ ]:
def binarize(gray):
    block = 25 if gray.shape[0] > 40 else 11
    if block % 2 == 0:
        block += 1
    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=block,
        C=5,
    )
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    return bw

### Visualització: pre-procés + deskew + binarització

Tres columnes per veure el resultat de cada etapa sobre la mostra anterior.

In [ ]:
fig, axes = plt.subplots(N_SHOW, 3, figsize=(14, 2.2 * N_SHOW))
for i, p in enumerate(sample_paths):
    crop = cv2.imread(str(p))
    pre  = preprocess(crop)
    skew = deskew(pre)
    bw   = binarize(skew)

    axes[i, 0].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(p.name, fontsize=8); axes[i, 0].axis('off')

    axes[i, 1].imshow(skew); axes[i, 1].set_title('pre + deskew', fontsize=9); axes[i, 1].axis('off')

    axes[i, 2].imshow(bw);   axes[i, 2].set_title('binaritzada',  fontsize=9); axes[i, 2].axis('off')

plt.tight_layout(); plt.show()

## 5. Segmentació per projecció vertical

Sumem píxels blancs per columna → histograma. Les columnes de **gap** entre caràcters tenen valors baixos. Detectem les **valls** (seqüències de columnes amb projecció normalitzada < `VALLEY_FRAC`) i agafem el centre de cada vall com a tall.

Filtre de separació mínima per evitar **over-segmentació** (tallar dins d'un caràcter com el '0' o la 'D').

In [ ]:
VALLEY_FRAC = 0.10   # llindar de vall (fracció del màxim)
MIN_GAP_FRAC = 0.3   # separació mínima entre talls (× h del retall)

def projection_cuts(bw):
    h, w = bw.shape
    proj = bw.sum(axis=0).astype(float)
    if proj.max() == 0:
        return []
    proj_n = proj / proj.max()
    # Suavitzat lleuger per eliminar pics
    proj_n = np.convolve(proj_n, np.ones(3) / 3, mode='same')

    below   = proj_n < VALLEY_FRAC
    changes = np.diff(below.astype(np.int8), prepend=0, append=0)
    starts  = np.where(changes ==  1)[0]
    ends    = np.where(changes == -1)[0]
    raw     = [(int(s) + int(e)) // 2 for s, e in zip(starts, ends)]

    # Filtre de separació mínima
    min_gap = max(3, int(h * MIN_GAP_FRAC))
    cuts = []
    for c in raw:
        if not cuts or c - cuts[-1] > min_gap:
            cuts.append(c)
    return cuts

## 6. Fallback: components connexos

Quan els caràcters es toquen o el fons és sorollós, la projecció no troba valls clars. En aquest cas detectem els caràcters com a **components connexos** filtrats per:

- Alçada ≥ 40 % de l'alçada del retall (descarta restes de marc).
- Aspect ratio entre 0.15 i 1.2 (caràcters són estrets).
- Densitat mínima dins la seva bounding box.

Els ordenem d'esquerra a dreta.

In [ ]:
def cc_strips(bw, min_h_frac=0.4, ar_min=0.15, ar_max=1.2, fill_min=0.15):
    h_roi = bw.shape[0]
    n, _, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
    cands = []
    for i in range(1, n):  # 0 = fons
        x, y, w, h, area = stats[i]
        if h < min_h_frac * h_roi:        continue
        if w == 0:                         continue
        ar = w / h
        if not (ar_min <= ar <= ar_max):   continue
        if area < fill_min * w * h:        continue
        cands.append((x, y, w, h))
    cands.sort(key=lambda b: b[0])  # ordre L→R
    return [(x, bw[y:y + h, x:x + w]) for (x, y, w, h) in cands]

## 7. Talls → tires de caràcter

Donada la llista de talls (projecció) o de bounding boxes (CC), extreiem una tira binària per cada caràcter i li retallem les files buides dalt i baix per centrar-lo.

In [ ]:
def cuts_to_strips(bw, cuts):
    h, w = bw.shape
    bounds = sorted(set([0, *cuts, w]))
    out = []
    for a, b in zip(bounds, bounds[1:]):
        if b - a < 3:
            continue
        strip = bw[:, a:b]
        rows  = strip.sum(axis=1)
        nz    = np.where(rows > 0)[0]
        if len(nz) == 0:
            continue
        r0 = max(0,    nz[0]  - 1)
        r1 = min(h,    nz[-1] + 2)
        out.append((a, strip[r0:r1, :]))
    return out

## 8. Normalització a 64×32

Cada caràcter es redimensiona preservant l'aspect ratio i s'enquadra en una *canvas* fixa de 64×32 amb padding negre. Aquesta és la mida estàndard que farà servir el classificador.

In [ ]:
CHAR_H, CHAR_W = 64, 32

def normalize_char(ch):
    h, w = ch.shape
    if h == 0 or w == 0:
        return np.zeros((CHAR_H, CHAR_W), np.uint8)
    scale = min(CHAR_H / h, CHAR_W / w)
    nh, nw = max(1, int(h * scale)), max(1, int(w * scale))
    resized = cv2.resize(ch, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((CHAR_H, CHAR_W), np.uint8)
    yo = (CHAR_H - nh) // 2
    xo = (CHAR_W - nw) // 2
    canvas[yo:yo + nh, xo:xo + nw] = resized
    return canvas

## 9. Pipeline complet + validació

Encadenem tot. La **validació** és el que ens permet rebutjar els fals positius del detector:

- Nombre de caràcters segmentats dins `[N_MIN, N_MAX]` (matrícules EU: 5–8).
- Alçades dels caràcters consistents: `std/mean < 0.35`.

Si no compleix, retorna `None` i el retall queda marcat com a *no-matrícula*.

In [ ]:
N_MIN, N_MAX = 3, 12
HEIGHT_STD_MAX = 0.35

def segment_plate(crop_bgr):
    pre  = preprocess(crop_bgr)
    skew = deskew(pre)
    bw   = binarize(skew)

    # Intent 1: projecció vertical
    strips = cuts_to_strips(bw, projection_cuts(bw))

    # Intent 2: fallback per CC si la projecció no funciona
    if not (N_MIN <= len(strips) <= N_MAX):
        strips = cc_strips(bw)

    if not (N_MIN <= len(strips) <= N_MAX):
        return {'ok': False, 'reason': f'n_chars={len(strips)}', 'bw': bw, 'strips': strips, 'chars': None}

    heights = [s.shape[0] for _, s in strips]
    if np.mean(heights) > 0 and np.std(heights) / np.mean(heights) > HEIGHT_STD_MAX:
        return {'ok': False, 'reason': 'height_var', 'bw': bw, 'strips': strips, 'chars': None}

    chars = [normalize_char(s) for _, s in strips]
    return {'ok': True, 'reason': 'ok', 'bw': bw, 'strips': strips, 'chars': chars}

## 10. Visualització del pipeline sobre una mostra

Per cada retall: original, binaritzada (amb número de tires trobades) i els caràcters normalitzats concatenats (o "rebutjat" en vermell si ha fallat la validació).

In [ ]:
viz_paths = random.sample(crops, 15)
fig, axes = plt.subplots(len(viz_paths), 3, figsize=(16, 1.8 * len(viz_paths)))
for i, p in enumerate(viz_paths):
    crop = cv2.imread(str(p))
    res  = segment_plate(crop)

    axes[i, 0].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(p.name, fontsize=8); axes[i, 0].axis('off')

    axes[i, 1].imshow(res['bw'])
    axes[i, 1].set_title(f"bin — {len(res['strips'])} tires", fontsize=9)
    axes[i, 1].axis('off')

    if res['ok']:
        merged = np.hstack(res['chars'])
        axes[i, 2].imshow(merged)
        axes[i, 2].set_title(f"OK — {len(res['chars'])} chars", fontsize=9, color='green')
    else:
        axes[i, 2].text(0.5, 0.5, f"REBUTJAT\n({res['reason']})",
                        ha='center', va='center', fontsize=10, color='red',
                        transform=axes[i, 2].transAxes)
    axes[i, 2].axis('off')
plt.tight_layout(); plt.show()

## 11. Estadístiques globals

Apliquem el pipeline a **tots** els retalls i mirem quants passen la validació. És el primer indicador de qualitat del pre-procés: si rebutgem ~70-80 % (els fals positius del detector) i acceptem la resta amb 6-8 caràcters, anem ben encaminats.

In [ ]:
from collections import Counter

n_ok = 0
reasons = Counter()
n_chars_dist = Counter()

for p in crops:
    crop = cv2.imread(str(p))
    if crop is None:
        reasons['unreadable'] += 1
        continue
    res = segment_plate(crop)
    if res['ok']:
        n_ok += 1
        n_chars_dist[len(res['chars'])] += 1
    else:
        reasons[res['reason']] += 1

print(f'Total crops:     {len(crops)}')
print(f'Acceptats:       {n_ok} ({100 * n_ok / len(crops):.1f}%)')
print(f'Rebutjats:       {len(crops) - n_ok}')
print('\nMotius de rebuig:')
for r, c in reasons.most_common():
    print(f'  {r:20s} {c}')
print('\nDistribució de #caràcters dels acceptats:')
for k in sorted(n_chars_dist):
    print(f'  {k} chars: {n_chars_dist[k]}')

## 12. Següents passos

- Si la taxa d'acceptació és **baixa per `n_chars`** → afluixar `VALLEY_FRAC` o `MIN_GAP_FRAC`, o millorar la binarització (provar Sauvola).
- Si rebutja **per `height_var`** → revisar el deskew i el filtre CC (alçada mínima).
- Si accepta **massa fals positius** → afegir més validacions (regex de format, ratio width/height global del retall).
- Quan el pre-procés sigui sòlid → notebook 03: dataset sintètic + entrenament HOG+SVM.